In [ ]:
from difflib import get_close_matches
import re

def auto_correct_question(valid_values, question):
    words = re.findall(r'\b\w+\b', question)  # Extract words using regex
    corrected_question = []
    suggestions = {}
    
    for i in range(len(words)):
        best_match = None
        best_matches = []
        for phrase in valid_values:
            phrase_words = phrase.split()
            if len(phrase_words) > 1:
                phrase_pattern = re.compile(r'\b' + re.escape(phrase) + r'\b', re.IGNORECASE)
                if phrase_pattern.search(question):
                    best_match = phrase
                    best_matches.append(phrase)
                    break
            
        if not best_match:
            matches = get_close_matches(words[i], valid_values, n=3, cutoff=0.6)
            print(matches)
            if matches:
                best_match = matches[0]
                best_matches = matches
            else:
                best_match = words[i]
        
        corrected_question.append(best_match)
        if len(best_matches) > 1:
            suggestions[words[i]] = best_matches
    
    return ' '.join(corrected_question), suggestions


In [34]:
def auto_correct_question(valid_values, question):
    words = re.findall(r'\b\w+\b', question)  # Extract words using regex
    corrected_question = []
    suggestions = {}
    
    for i in range(len(words)):
        best_match = None
        best_matches = []
        for phrase in valid_values:
            phrase_words = phrase.split()
            for j in range(len(phrase_words)):
                if words[i] == phrase_words[j]:  # Check if part of a phrase matches
                    best_matches.append(phrase)
                    
        if best_matches:
            best_match = best_matches[0]  # Pick the first match as default
        else:
            matches = get_close_matches(words[i], valid_values, n=3, cutoff=0.6)
            if matches:
                best_match = matches[0]
                best_matches.extend(matches)
            else:
                best_match = words[i]
        
        corrected_question.append(best_match)
        if len(best_matches) > 1:
            suggestions[words[i]] = best_matches
    
    return ' '.join(corrected_question), suggestions

In [76]:

# Example list of valid values
valid_values = [
    "agile developer", "python lead", "artificial intelligence", "software developer", "python developer", "ml engineer", "data scientist",
    "data analyst", "data engineer", "data architect", "data manager", "data steward", "data modeler", "security intelligence",
    "data administrator", "data security analyst", "data security manager", "data security administrator", "data security architect",
]


In [77]:

# Example question
question = "How many Manager resources are there with inteligence"


In [78]:

corrected_question, suggestions = auto_correct_question(valid_values, question.lower())
print("Corrected Question:", corrected_question)


Corrected Question: how many data manager resources are there with security intelligence


In [79]:

if suggestions:
    print("Suggestions for ambiguous words:")
    for word, options in suggestions.items():
        print(f"{word}: {options}")

Suggestions for ambiguous words:
manager: ['data manager', 'data security manager']
inteligence: ['security intelligence', 'artificial intelligence']


In [2]:
from fuzzywuzzy import fuzz, process

db_columns = {
    "ProjServiceArea": "Project Service Area",
    "Proj/SendDept": "Project Send Department",
    "PlanningGB": "Planning GB"
}

db_column_values = {
    "SDS/EN5285": {"ProjServiceArea", "Proj/SendDept"},
    "EI-1032384": {"PlanningGB", "Proj/SendDept"}
}

db_valid_values = [
    "agile developer", "python lead", "artificial intelligence", "software developer", "python developer", "ml engineer", "data scientist",
    "data analyst", "data engineer", "data architect", "data manager", "data steward", "data modeler", "security intelligence",
    "data administrator", "data security analyst", "data security manager", "data security administrator", "data security architect"
]

h:\Miniconda\envs\GenAI\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
def fuzzy_match_and_correct(question, choices, threshold=80):
    words = question.split()
    corrected_words = []
    
    for i in range(len(words)):
        match, score = process.extractOne(words[i], choices) if words[i] else (None, 0)
        if score >= threshold:
            corrected_words.append(match)
        else:
            corrected_words.append(words[i])
    
    return " ".join(corrected_words)

In [4]:
def process_question(question):
    # Step 1: Fuzzy match and correct db_valid_values
    corrected_question = fuzzy_match_and_correct(question, db_valid_values)
    
    # Step 2: Fuzzy match and correct db_columns
    corrected_question = fuzzy_match_and_correct(corrected_question, db_columns.values())
    
    # Step 3: Fuzzy match and correct db_column_values
    corrected_question = fuzzy_match_and_correct(corrected_question, db_column_values.keys())
    
    return corrected_question

In [4]:
question = "How many Manager resources are there with inteligence"
corrected_question = process_question(question)
print("Corrected Question:", corrected_question)

Corrected Question: How many data manager resources software developer there with artificial intelligence


In [19]:
from fuzzywuzzy import fuzz, process

db_columns = {
    "ProjServiceArea": "Project Service Area",
    "Proj/SendDept": "Project Send Department",
    "PlanningGB": "Planning GB",
}

db_column_values = {
    "SDS/EN5285": {"ProjServiceArea", "Proj/SendDept"},
    "EI-1032384": {"PlanningGB", "Proj/SendDept"},
}

db_valid_values = [
    "agile developer", "python lead", "artificial intelligence", "software developer", "python developer", "ml engineer", "data scientist",
    "data analyst", "data engineer", "data architect", "data manager", "data steward", "data modeler", "security intelligence",
    "data administrator", "data security analyst", "data security manager", "data security administrator", "data security architect",
]

In [6]:
def fuzzy_match(query, choices, threshold=60):
    matches = process.extract(query, choices, limit=3)
    matches = [match for match in matches if match[1] >= threshold]
    if len(matches) > 1:
        print(f"Multiple matches found for '{query}': {[m[0] for m in matches]}")
        return 
    elif matches:
        return matches[0][0]
    return query

In [7]:
def process_question(question):
    words = question.split()
    corrected_words = []
    
    # First, correct skill names
    for i in range(len(words)):
        phrase = " ".join(words[i:i+3])
        corrected = fuzzy_match(phrase, db_valid_values)
        if corrected != phrase:
            corrected_words.append(corrected)
            i += 2  # Skip the next two words since they are already considered
        else:
            corrected_words.append(words[i])
    
    corrected_question = " ".join(corrected_words)
    
    # Second, correct database column names
    for key in db_columns.keys():
        corrected_question = corrected_question.replace(key, fuzzy_match(key, db_columns.keys()))
    
    # Third, correct database column values
    for key in db_column_values.keys():
        corrected_question = corrected_question.replace(key, fuzzy_match(key, db_column_values.keys()))
    
    print("Corrected Question:", corrected_question)
    return corrected_question

In [ ]:
# Example usage
question = "What is the total employee count in SDS/EN5385 with skills in pyton?"
process_question(question)


In [11]:

def fuzzy_match(query, choices, threshold=70, min_length=2):  # Lowered min_length to handle short words like 'pyton'
    if len(query) < min_length:
        return query  # Skip matching if the word is too short
    
    match = process.extractOne(query, choices, scorer=fuzz.token_sort_ratio)  # More robust matching
    if match and match[1] >= threshold:
        return match[0]
    return query

In [24]:

def process_question(question):
    question = question.replace("?", " ?")
    question = question.replace(".", " .")

    words = question.split()
    corrected_words = []
    
    # First, correct skill names in db_valid_values (considering 1, 2, or 3-word phrases)
    i = 0
    while i < len(words):
        best_match = words[i]
        best_score = 0
        best_phrase = words[i]
        
        for j in range(1, 4):  # Consider phrases of length 1, 2, or 3
            if i + j <= len(words):
                phrase = " ".join(words[i:i + j])
                if len(phrase) < 4:  # Avoid short words
                    continue
                match = process.extract(phrase, db_valid_values,limit=5)
                if match:
                    print(phrase)
                    print(match)
                    best_score = 80 
                # if match and match[1] > best_score and match[1] >= 70:
                #     best_match, best_score = match[0], match[1]
                #     best_phrase = phrase
                #     print(best_phrase)
        
        if best_score >= 70:
            corrected_words.append(best_match)
            i += len(best_phrase.split())  # Move index forward by the matched phrase length
        else:
            corrected_words.append(words[i])
            i += 1  # Move to the next word
    
    corrected_question = " ".join(corrected_words)
    
    # Second, correct database column names (db_columns) using fuzzy matching
    for word in corrected_question.split():
        match = fuzzy_match(word, db_columns.keys())
        if match != word:
            corrected_question = corrected_question.replace(word, match)
    
    # Third, correct database column values (db_column_values) using fuzzy matching
    for word in corrected_question.split():
        match = fuzzy_match(word, db_column_values.keys(), threshold=70, min_length=6)  # Lower threshold for IDs
        if match != word:
            corrected_question = corrected_question.replace(word, match)
    
    # Prompt for ambiguous matches in db_column_values
    for key, values in db_column_values.items():
        if key in corrected_question and len(values) > 1:
            print(f"{key} is associated with {values}, which one do you want to select?")
            selected = input("Please enter your choice: ")
            corrected_question = corrected_question.replace(key, selected)
    
    print("Corrected Question:", corrected_question)
    return corrected_question






In [26]:
# Example usage
question = "What is the total employee count in SDS/EN5385 with skills in pyton?"
process_question(question)

What
[('artificial intelligence', 45), ('software developer', 45), ('data scientist', 45), ('data analyst', 45), ('data engineer', 45)]
What is
[('data scientist', 51), ('data steward', 51), ('data security analyst', 51), ('data security manager', 51), ('data security administrator', 51)]
What is the
[('data steward', 52), ('data scientist', 48), ('data architect', 48), ('data analyst', 43), ('data engineer', 42)]
is the
[('data scientist', 60), ('data architect', 50), ('agile developer', 45), ('artificial intelligence', 45), ('data steward', 45)]
is the total
[('data administrator', 45), ('data security administrator', 45), ('security intelligence', 38), ('python lead', 35), ('software developer', 33)]
the total
[('data architect', 48), ('data security architect', 48), ('python lead', 40), ('data administrator', 40), ('data security analyst', 40)]
the total employee
[('data modeler', 52), ('ml engineer', 43), ('python developer', 41), ('python lead', 40), ('artificial intelligence', 3

KeyboardInterrupt: Interrupted by user

In [27]:
from textblob import TextBlob

def correct_spelling(word):
    return str(TextBlob(word).correct())

In [29]:
correct_spelling("pthon")

'then'

In [34]:
import importlib.resources

from symspellpy import SymSpell, Verbosity

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = f"H:\\Miniconda\\envs\\GenAI\\Lib\\site-packages\\symspellpy\\frequency_dictionary_en_82_765.txt"
# term_index is the column of the term and count_index is the
# column of the term frequency
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

# lookup suggestions for single-word input strings
input_term = "aapliction"  # misspelling of "members"
# max edit distance per lookup
# (max_edit_distance_lookup <= max_dictionary_edit_distance)
suggestions = sym_spell.lookup(input_term, Verbosity.CLOSEST, max_edit_distance=2)
# display suggestion term, edit distance, and term frequency
for suggestion in suggestions:
    print(suggestion)

application, 2, 152776595
affliction, 2, 546664


In [ ]:
valid_values = [
    "agile developer", "python lead", "artificial intelligence", "software developer", "python developer", "ml engineer", "data scientist",
    "data analyst", "data engineer", "data architect", "data manager", "data steward", "data modeler", "security intelligence",
    "data administrator", "data security analyst", "data security manager", "data security administrator", "data security architect",
]

In [ ]:
db_column_values = {
'Base S1267' : {'Cluster'},
'Base S4991' : {'PrimSkillSet'},
'Cloud 5018' : {'Cluster'},
'Cloud 9084' : {'PrimSkillSet'},
'Data E4353' : {'PrimSkillSet'},
'Data S4776' : {'Cluster'},
'Data S7310' : {'PrimSkillSet'},
'HMI Ap7679' : {'PrimSkillSet'},
'HMI De4852' : {'PrimSkillSet'},
'HMI De7207' : {'Cluster'},
'IoT Ap7714' : {'PrimSkillSet'},
'IT Bus9295' : {'PrimSkillSet'},
'IT Inf5951' : {'PrimSkillSet'},
'IT Lab2382' : {'PrimSkillSet'},
'ITeS &6436' : {'Cluster'},
'ITeS -3262' : {'PrimSkillSet'},
'ITeS -5034' : {'PrimSkillSet'},
'ITeS -8836' : {'PrimSkillSet'},
'ITeS -9167' : {'PrimSkillSet'},
'Linux 6464' : {'PrimSkillSet'},
'Model 7170' : {'Cluster'},
'New Ag6409' : {'Cluster'},
'SIL - 8149' : {'PrimSkillSet'},
}